# PCA Diagnostic Plots: A Comprehensive Guide
## Using European City Temperature Data

This notebook demonstrates all major PCA diagnostic and interpretation plots using the CityTemp dataset, which contains monthly average temperatures for 33 European cities.

**Learning Objectives:**
- Understand how to interpret each PCA plot
- Learn to extract insights from multivariate data
- Master the workflow from data exploration to model diagnostics

---

## Table of Contents

1. [Setup and Data Loading](#1.-Setup-and-Data-Loading)
2. [Model Interpretation Plots](#2.-Model-Interpretation-Plots)
   - 2.1 Score Plot
   - 2.2 Loading Plot
   - 2.3 Biplot
   - 2.4 Correlation Loading Plot
   - 2.5 Score Line Plot
   - 2.6 3D Score Plot
3. [Model Validation & Selection](#3.-Model-Validation-&-Selection)
   - 3.1 Scree Plot
   - 3.2 Explained Variance Plot
   - 3.3 Cross-Validation Plot
4. [Outlier Detection & Diagnostics](#4.-Outlier-Detection-&-Diagnostics)
   - 4.1 Hotelling's T² Statistic
   - 4.2 Q Statistic (SPE)
   - 4.3 T² vs Q Diagnostic Plot
   - 4.4 Influence Plot
5. [Fault Diagnosis / Root Cause](#5.-Fault-Diagnosis-/-Root-Cause)
   - 5.1 T² Contribution Plot
   - 5.2 Q Contribution Plot
   - 5.3 Score Contribution Plot
6. [Variable Analysis](#6.-Variable-Analysis)
   - 6.1 Loading Bar Chart
   - 6.2 Loading Heatmap
   - 6.3 Variable Importance Plot
   - 6.4 Communalities Plot
7. [Residual Analysis](#7.-Residual-Analysis)
   - 7.1 Residual Matrix Heatmap
   - 7.2 Predicted vs Actual
   - 7.3 Residual Histogram
   - 7.4 Q-Q Plot
8. [Summary Dashboard](#8.-Summary-Dashboard)

---
## 1. Setup and Data Loading

In [28]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import f as f_dist
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')

# Color palette
COLORS = {
    'score': '#4169E1',      # Royal Blue
    'loading': '#DC143C',     # Crimson
    'normal': '#228B22',      # Forest Green
    'warning': '#FF8C00',     # Dark Orange
    'outlier': '#8B0000',     # Dark Red
    'grid': '#DCDCDC'         # Light Gray
}

print("✓ Libraries loaded successfully!")

✓ Libraries loaded successfully!


In [29]:
# Load the data
df = pd.read_excel('./data/CityTemp.xlsx')

print(f"Dataset shape: {df.shape}")
print(f"\nCities ({len(df)}): {', '.join(df['Cities'].tolist())}")
print(f"\nVariables: {df.columns.tolist()[1:]}")
df.head(10)

Dataset shape: (33, 14)

Cities (33): Amsterdam, Ankara, Athens, Belgrade, Berlin, Bern, Bonn, Bruxells, Bucarest, Budapest, Copenhagen, Dublin, Helsinki, Lisbon, London, Luxemborg, Marbella, Moscow, Oslo, Paris, Prague, Rome, Sofia, Stockholm, Vienna, Warsaw, Bratislava, St. Petersberg, Ljubljana, Madrid, Milan, Edinburgh, Skopje

Variables: ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Latitude']


,Cities,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec,Latitude
0,Amsterdam,2.0,2.0,5.0,8.0,12.0,15.0,17.0,17.0,15.0,11.0,6.0,3.0,52.37
1,Ankara,1.0,0.0,6.0,12.0,17.0,19.0,23.0,22.0,18.0,14.0,8.0,2.0,39.87
2,Athens,9.0,10.0,12.0,16.0,21.0,26.0,29.0,28.0,24.0,20.0,15.0,11.0,37.97
3,Belgrade,1.0,3.0,8.0,13.0,17.0,22.0,23.0,23.0,19.0,14.0,8.0,6.0,44.82
4,Berlin,-2.0,1.0,4.0,10.0,14.0,18.0,21.0,19.0,16.0,10.0,4.0,0.0,52.52
5,Bern,1.0,1.0,6.0,10.0,13.0,17.0,18.0,19.0,15.0,10.0,4.0,1.0,46.95
6,Bonn,1.0,2.0,7.0,11.0,14.0,18.0,20.0,16.0,16.0,11.0,7.0,2.0,50.73
7,Bruxells,1.0,4.0,7.0,11.0,13.0,18.0,19.0,18.0,17.0,12.0,7.0,3.0,50.85
8,Bucarest,-4.0,-2.0,6.0,12.0,17.0,21.0,23.0,23.0,18.0,13.0,7.0,1.0,44.42
9,Budapest,-2.0,1.0,7.0,13.0,17.0,21.0,23.0,23.0,18.0,13.0,6.0,1.0,47.47


In [30]:
# Prepare data for PCA
# Use only temperature variables (Jan-Dec), exclude Latitude for now
city_names = df['Cities'].values
month_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
X_raw = df[month_cols].values
latitude = df['Latitude'].values

# Standardize the data (mean=0, std=1)
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f"Data matrix X shape: {X.shape}")
print(f"  - {X.shape[0]} samples (cities)")
print(f"  - {X.shape[1]} variables (months)")
print(f"\nData standardized: mean ≈ 0, std ≈ 1")

Data matrix X shape: (33, 12)
  - 33 samples (cities)
  - 12 variables (months)

Data standardized: mean ≈ 0, std ≈ 1


In [31]:
# Fit PCA model
n_components = X.shape[1]  # Keep all components initially
pca = PCA(n_components=n_components)
T = pca.fit_transform(X)  # Scores
P = pca.components_.T      # Loadings (transposed to match convention)
eigenvalues = pca.explained_variance_
explained_var_ratio = pca.explained_variance_ratio_

print("PCA Model Fitted!")
print(f"\nScores matrix T shape: {T.shape}")
print(f"Loadings matrix P shape: {P.shape}")
print(f"\nEigenvalues: {np.round(eigenvalues, 3)}")
print(f"\nExplained variance ratio: {np.round(explained_var_ratio * 100, 1)}%")
print(f"Cumulative: {np.round(np.cumsum(explained_var_ratio) * 100, 1)}%")

PCA Model Fitted!

Scores matrix T shape: (33, 12)
Loadings matrix P shape: (12, 12)

Eigenvalues: [1.0521e+01 1.2600e+00 2.6700e-01 1.0900e-01 7.3000e-02 5.2000e-02
 3.1000e-02 1.8000e-02 1.6000e-02 1.4000e-02 9.0000e-03 5.0000e-03]

Explained variance ratio: [85.  10.2  2.2  0.9  0.6  0.4  0.2  0.1  0.1  0.1  0.1  0. ]%
Cumulative: [ 85.   95.2  97.4  98.2  98.8  99.2  99.5  99.6  99.8  99.9 100.  100. ]%


### Helper Functions for PCA Diagnostics

In [32]:
def compute_hotelling_t2(scores, eigenvalues, n_components):
    """Compute Hotelling's T² statistic for each sample."""
    T2 = np.sum((scores[:, :n_components] ** 2) / eigenvalues[:n_components], axis=1)
    return T2

def compute_t2_limit(n_samples, n_components, alpha=0.05):
    """Compute T² control limit using F-distribution."""
    f_crit = f_dist.ppf(1 - alpha, n_components, n_samples - n_components)
    t2_limit = (n_components * (n_samples - 1) / (n_samples - n_components)) * f_crit
    return t2_limit

def compute_q_statistic(X, scores, loadings, n_components):
    """Compute Q statistic (SPE) for each sample."""
    X_reconstructed = scores[:, :n_components] @ loadings[:, :n_components].T
    E = X - X_reconstructed
    Q = np.sum(E ** 2, axis=1)
    return Q, E

def compute_q_limit(eigenvalues, n_components, alpha=0.05):
    """Compute Q control limit using Jackson-Mudholkar approximation."""
    theta1 = np.sum(eigenvalues[n_components:])
    theta2 = np.sum(eigenvalues[n_components:] ** 2)
    theta3 = np.sum(eigenvalues[n_components:] ** 3)
    
    if theta1 == 0:
        return 0
    
    h0 = 1 - (2 * theta1 * theta3) / (3 * theta2 ** 2)
    c_alpha = stats.norm.ppf(1 - alpha)
    
    q_limit = theta1 * (c_alpha * np.sqrt(2 * theta2 * h0 ** 2) / theta1 + 
                        1 + theta2 * h0 * (h0 - 1) / theta1 ** 2) ** (1 / h0)
    return q_limit

def compute_t2_contributions(X, scores, loadings, eigenvalues, n_components):
    """Compute T² contributions for each variable."""
    contributions = np.zeros_like(X)
    for k in range(n_components):
        contributions += (scores[:, k:k+1] * loadings[:, k]) / eigenvalues[k]
    contributions = X * contributions
    return contributions

def compute_q_contributions(E):
    """Compute Q contributions for each variable (squared residuals)."""
    return E ** 2

print("✓ Helper functions defined!")

✓ Helper functions defined!


---
## 2. Model Interpretation Plots

These plots help us understand the structure in our data - how samples relate to each other and how variables contribute to the principal components.

### 2.1 Score Plot ($t_1$ vs $t_2$)

**Purpose:** Visualize sample (city) relationships in reduced dimensionality.

**What to look for:**
- **Clusters:** Groups of similar cities
- **Gradients:** Continuous trends (e.g., latitude effect)
- **Outliers:** Cities with unusual temperature patterns

In [33]:
fig = go.Figure()

# Add scatter plot colored by latitude
fig.add_trace(go.Scatter(
    x=T[:, 0],
    y=T[:, 1],
    mode='markers+text',
    text=city_names,
    textposition='top right',
    marker=dict(
        size=12,
        color=latitude,
        colorscale='RdBu_r',
        showscale=True,
        colorbar=dict(title='Latitude (°N)'),
        line=dict(color='black', width=0.5)
    ),
    hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>Latitude: %{marker.color:.1f}°N<extra></extra>'
))

# Add reference lines
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=1)

fig.update_layout(
    title='Score Plot: European Cities by Temperature Pattern',
    xaxis_title=f'PC1 ({explained_var_ratio[0]*100:.1f}% variance)',
    yaxis_title=f'PC2 ({explained_var_ratio[1]*100:.1f}% variance)',
    width=1000,
    height=700,
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- PC1 separates cities by overall temperature (warm Mediterranean vs cold Nordic)")
print("- The color gradient shows latitude strongly correlates with PC1")
print("- Cities clustered together have similar annual temperature patterns")


📊 INTERPRETATION:
- PC1 separates cities by overall temperature (warm Mediterranean vs cold Nordic)
- The color gradient shows latitude strongly correlates with PC1
- Cities clustered together have similar annual temperature patterns


### 2.2 Loading Plot ($p_1$ vs $p_2$)

**Purpose:** Understand how variables (months) contribute to each PC and their relationships.

**What to look for:**
- **Variable groupings:** Months that cluster together are correlated
- **Distance from origin:** Important variables are far from center
- **Angles:** ~0° = positive correlation, ~90° = no correlation, ~180° = negative correlation

In [34]:
fig = go.Figure()

# Plot loadings as arrows
for i, month in enumerate(month_cols):
    # Arrow line
    fig.add_trace(go.Scatter(
        x=[0, P[i, 0]],
        y=[0, P[i, 1]],
        mode='lines+text',
        line=dict(color='crimson', width=2),
        text=['', month],
        textposition='top center',
        textfont=dict(size=11),
        showlegend=False,
        hovertemplate=f'<b>{month}</b><br>PC1: {P[i, 0]:.3f}<br>PC2: {P[i, 1]:.3f}<extra></extra>'
    ))
    # Marker at end
    fig.add_trace(go.Scatter(
        x=[P[i, 0]],
        y=[P[i, 1]],
        mode='markers',
        marker=dict(size=12, color='crimson', line=dict(color='black', width=1)),
        showlegend=False,
        hoverinfo='skip'
    ))

# Add unit circle
theta = np.linspace(0, 2*np.pi, 100)
fig.add_trace(go.Scatter(
    x=np.cos(theta),
    y=np.sin(theta),
    mode='lines',
    line=dict(color='gray', dash='dash', width=1),
    showlegend=False,
    hoverinfo='skip'
))

# Add reference lines
fig.add_hline(y=0, line_dash='solid', line_color='gray', line_width=0.5)
fig.add_vline(x=0, line_dash='solid', line_color='gray', line_width=0.5)

fig.update_layout(
    title='Loading Plot: Monthly Temperature Contributions',
    xaxis_title=f'PC1 Loading ({explained_var_ratio[0]*100:.1f}%)',
    yaxis_title=f'PC2 Loading ({explained_var_ratio[1]*100:.1f}%)',
    width=900,
    height=750,
    xaxis=dict(range=[-0.5, 0.5], scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[-0.8, 0.8]),
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- All months load positively on PC1 → PC1 represents 'average temperature level'")
print("- Summer months (Jun-Aug) vs Winter months (Dec-Feb) separate on PC2")
print("- PC2 captures 'seasonality' or 'continental vs maritime' climate")
print("- Adjacent months cluster together (Mar-Apr-May, etc.)")


📊 INTERPRETATION:
- All months load positively on PC1 → PC1 represents 'average temperature level'
- Summer months (Jun-Aug) vs Winter months (Dec-Feb) separate on PC2
- PC2 captures 'seasonality' or 'continental vs maritime' climate
- Adjacent months cluster together (Mar-Apr-May, etc.)


### 2.3 Biplot: Scores and Loadings Combined

**Purpose:** See both samples and variables in one plot to understand which variables drive sample positions.

**Reading rules:**
- Cities in the direction of a month arrow have high values for that month
- Cities opposite an arrow have low values for that variable

In [35]:
# Scale factor for loadings (to make them visible alongside scores)
scale = 3

fig = go.Figure()

# Plot scores (cities) with latitude color
fig.add_trace(go.Scatter(
    x=T[:, 0],
    y=T[:, 1],
    mode='markers+text',
    text=city_names,
    textposition='top right',
    textfont=dict(size=8),
    marker=dict(
        size=10,
        color=latitude,
        colorscale='RdBu_r',
        showscale=True,
        colorbar=dict(title='Latitude (°N)'),
        line=dict(color='black', width=0.5),
        opacity=0.7
    ),
    name='Cities (scores)',
    hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
))

# Plot loadings (months) as arrows
for i, month in enumerate(month_cols):
    # Arrow line
    fig.add_trace(go.Scatter(
        x=[0, P[i, 0]*scale],
        y=[0, P[i, 1]*scale],
        mode='lines',
        line=dict(color=COLORS['loading'], width=2),
        showlegend=False,
        hoverinfo='skip'
    ))
    # Arrow head (annotation)
    fig.add_annotation(
        x=P[i, 0]*scale,
        y=P[i, 1]*scale,
        ax=0,
        ay=0,
        xref='x',
        yref='y',
        axref='x',
        ayref='y',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor=COLORS['loading']
    )
    # Month label
    fig.add_annotation(
        x=P[i, 0]*scale,
        y=P[i, 1]*scale,
        text=month,
        showarrow=False,
        font=dict(size=10, color=COLORS['loading']),
        xshift=5,
        yshift=5
    )

# Add reference lines
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.8)
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.8)

# Add legend trace for loadings
fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(color=COLORS['loading'], width=2),
    name='Months (loadings)'
))

fig.update_layout(
    title='Biplot: Cities and Monthly Temperature Variables',
    xaxis_title=f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
    yaxis_title=f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
    width=1000,
    height=700,
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- Athens, Lisbon, Madrid (right side) → high temperatures year-round")
print("- Helsinki, Oslo, Stockholm (left side) → cold temperatures year-round")
print("- Cities along summer month arrows have hot summers (continental)")
print("- The biplot reveals climate patterns in one view!")


📊 INTERPRETATION:
- Athens, Lisbon, Madrid (right side) → high temperatures year-round
- Helsinki, Oslo, Stockholm (left side) → cold temperatures year-round
- Cities along summer month arrows have hot summers (continental)
- The biplot reveals climate patterns in one view!


### 2.4 Correlation Loading Plot

**Purpose:** Show how well each variable is explained by the model.

**The circles indicate:**
- **Outer circle (r=1):** 100% of variable variance explained
- **Inner circle (r=0.71):** 50% of variable variance explained
- Variables inside the inner circle need more components!

In [36]:
# Compute correlation loadings
# p_corr = p * sqrt(eigenvalue / var(x))
# Since X is standardized, var(x) = 1
P_corr = P * np.sqrt(eigenvalues)

fig = go.Figure()

# Draw circles using parametric equations
theta = np.linspace(0, 2*np.pi, 100)

# Outer circle (100%)
fig.add_trace(go.Scatter(
    x=np.cos(theta),
    y=np.sin(theta),
    mode='lines',
    line=dict(color='black', width=2),
    name='100% explained',
    hoverinfo='skip'
))

# Inner circle (50%)
fig.add_trace(go.Scatter(
    x=np.cos(theta) * np.sqrt(0.5),
    y=np.sin(theta) * np.sqrt(0.5),
    mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    name='50% explained',
    hoverinfo='skip'
))

# Plot correlation loadings - separate traces for well/poorly explained
well_explained_x = []
well_explained_y = []
well_explained_text = []
poorly_explained_x = []
poorly_explained_y = []
poorly_explained_text = []

for i, month in enumerate(month_cols):
    dist = np.sqrt(P_corr[i, 0]**2 + P_corr[i, 1]**2)
    if dist > np.sqrt(0.5):
        well_explained_x.append(P_corr[i, 0])
        well_explained_y.append(P_corr[i, 1])
        well_explained_text.append(month)
    else:
        poorly_explained_x.append(P_corr[i, 0])
        poorly_explained_y.append(P_corr[i, 1])
        poorly_explained_text.append(month)

# Add well explained points
fig.add_trace(go.Scatter(
    x=well_explained_x,
    y=well_explained_y,
    mode='markers+text',
    text=well_explained_text,
    textposition='top right',
    textfont=dict(size=11),
    marker=dict(
        size=15,
        color=COLORS['normal'],
        line=dict(color='black', width=1)
    ),
    name='Well explained (>50%)',
    hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>'
))

# Add poorly explained points
if poorly_explained_x:
    fig.add_trace(go.Scatter(
        x=poorly_explained_x,
        y=poorly_explained_y,
        mode='markers+text',
        text=poorly_explained_text,
        textposition='top right',
        textfont=dict(size=11),
        marker=dict(
            size=15,
            color=COLORS['warning'],
            line=dict(color='black', width=1)
        ),
        name='Poorly explained (<50%)',
        hovertemplate='<b>%{text}</b><br>PC1: %{x:.3f}<br>PC2: %{y:.3f}<extra></extra>'
    ))

# Add reference lines
fig.add_hline(y=0, line_dash='solid', line_color='gray', line_width=0.5)
fig.add_vline(x=0, line_dash='solid', line_color='gray', line_width=0.5)

# Add annotations for circle labels
fig.add_annotation(x=0.7, y=0.7, text='100%', showarrow=False, font=dict(size=10, color='gray'))
fig.add_annotation(x=0.45, y=0.45, text='50%', showarrow=False, font=dict(size=10, color='gray'))

fig.update_layout(
    title='Correlation Loading Plot',
    xaxis_title='Correlation with PC1',
    yaxis_title='Correlation with PC2',
    width=900,
    height=900,
    xaxis=dict(range=[-1.1, 1.1], scaleanchor='y', scaleratio=1),
    yaxis=dict(range=[-1.1, 1.1]),
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

# Calculate explained variance per variable
var_explained = P_corr[:, 0]**2 + P_corr[:, 1]**2

print("\n📊 INTERPRETATION:")
print("Variance explained by PC1+PC2 for each month:")
for i, month in enumerate(month_cols):
    status = "✓" if var_explained[i] > 0.5 else "⚠️"
    print(f"  {month}: {var_explained[i]*100:.1f}% {status}")


📊 INTERPRETATION:
Variance explained by PC1+PC2 for each month:
  Jan: 100.4% ✓
  Feb: 100.9% ✓
  Mar: 99.0% ✓
  Apr: 95.0% ✓
  May: 97.8% ✓
  Jun: 92.9% ✓
  Jul: 97.1% ✓
  Aug: 96.9% ✓
  Sep: 100.4% ✓
  Oct: 100.9% ✓
  Nov: 100.3% ✓
  Dec: 96.6% ✓


### 2.5 Score Line Plot

**Purpose:** Visualize scores for each sample as a profile, useful for seeing patterns across ordered samples.

In [37]:
# Sort cities by latitude for meaningful ordering
sort_idx = np.argsort(latitude)[::-1]  # North to South

fig = go.Figure()

x_pos = np.arange(len(city_names))

# PC1 trace
fig.add_trace(go.Scatter(
    x=x_pos,
    y=T[sort_idx, 0],
    mode='lines+markers',
    line=dict(color=COLORS['score'], width=2),
    marker=dict(size=8, symbol='circle'),
    name=f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
    hovertemplate='<b>%{text}</b><br>PC1: %{y:.2f}<extra></extra>',
    text=city_names[sort_idx]
))

# PC2 trace
fig.add_trace(go.Scatter(
    x=x_pos,
    y=T[sort_idx, 1],
    mode='lines+markers',
    line=dict(color=COLORS['loading'], width=2),
    marker=dict(size=8, symbol='square'),
    name=f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
    hovertemplate='<b>%{text}</b><br>PC2: %{y:.2f}<extra></extra>',
    text=city_names[sort_idx]
))

# Add reference line
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.8)

fig.update_layout(
    title='Score Line Plot: PC Scores by Latitude',
    xaxis_title='Cities (sorted by latitude: North → South)',
    yaxis_title='Score Value',
    width=1000,
    height=500,
    font=dict(size=12),
    xaxis=dict(
        tickmode='array',
        tickvals=x_pos,
        ticktext=city_names[sort_idx],
        tickangle=45
    ),
    hovermode='x unified'
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- PC1 shows clear latitude gradient (cold north, warm south)")
print("- PC2 captures deviations from this pattern (continental vs maritime effects)")
print("- Cities above/below the trend line have unique climate characteristics")


📊 INTERPRETATION:
- PC1 shows clear latitude gradient (cold north, warm south)
- PC2 captures deviations from this pattern (continental vs maritime effects)
- Cities above/below the trend line have unique climate characteristics


### 2.6 3D Score Plot

**Purpose:** Visualize three principal components when PC3 explains significant variance.

In [39]:
fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=T[:, 0],
    y=T[:, 1],
    z=T[:, 2],
    mode='markers+text',
    text=city_names,
    textposition='top center',
    textfont=dict(size=8),
    marker=dict(
        size=8,
        color=latitude,
        colorscale='RdBu_r',
        showscale=True,
        colorbar=dict(title='Latitude (°N)', len=0.5),
        line=dict(color='black', width=0.5)
    ),
    hovertemplate='<b>%{text}</b><br>PC1: %{x:.2f}<br>PC2: %{y:.2f}<br>PC3: %{z:.2f}<br>Latitude: %{marker.color:.1f}°N<extra></extra>'
))

fig.update_layout(
    title='3D Score Plot',
    scene=dict(
        xaxis_title=f'PC1 ({explained_var_ratio[0]*100:.1f}%)',
        yaxis_title=f'PC2 ({explained_var_ratio[1]*100:.1f}%)',
        zaxis_title=f'PC3 ({explained_var_ratio[2]*100:.1f}%)'
    ),
    width=900,
    height=800,
    font=dict(size=11)
)

fig.show()

print(f"\n📊 PC3 explains {explained_var_ratio[2]*100:.1f}% additional variance")
print("Use 3D plots when PC3 captures meaningful structure not visible in 2D.")


📊 PC3 explains 2.2% additional variance
Use 3D plots when PC3 captures meaningful structure not visible in 2D.


---
## 3. Model Validation & Selection

These plots help us decide how many principal components to retain.

### 3.1 Scree Plot

**Purpose:** Visual method for selecting number of components.

**Selection methods:**
- **Elbow method:** Look for where the curve flattens
- **Kaiser criterion:** Keep eigenvalues > 1

In [44]:
fig = go.Figure()

components = np.arange(1, len(eigenvalues) + 1)
n_selected = np.sum(eigenvalues > 1)

# Eigenvalue line
fig.add_trace(go.Scatter(
    x=components,
    y=eigenvalues,
    mode='lines+markers',
    line=dict(color=COLORS['score'], width=2),
    marker=dict(size=10),
    name='Eigenvalues',
    hovertemplate='PC%{x}<br>Eigenvalue: %{y:.3f}<extra></extra>'
))

# Highlight selected components (fill)
fig.add_trace(go.Scatter(
    x=np.concatenate([components[:n_selected], components[n_selected-1::-1]]),
    y=np.concatenate([eigenvalues[:n_selected], np.zeros(n_selected)]),
    fill='toself',
    fillcolor=COLORS['normal'],
    opacity=0.3,
    line=dict(width=0),
    name=f'Selected ({n_selected} PCs)',
    showlegend=True,
    hoverinfo='skip'
))

# Kaiser criterion line
fig.add_hline(y=1, line_dash='dash', line_color=COLORS['loading'], line_width=2, 
              annotation_text='Kaiser criterion (λ=1)', annotation_position='right')

fig.update_layout(
    title='Scree Plot',
    xaxis_title='Principal Component',
    yaxis_title='Eigenvalue',
    width=900,
    height=500,
    font=dict(size=12),
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    yaxis=dict(range=[0, max(eigenvalues) * 1.1]),
    hovermode='x unified'
)

fig.show()

print("\n📊 INTERPRETATION:")
print(f"Kaiser criterion suggests: {n_selected} components (eigenvalue > 1)")
print("\nEigenvalues:")
for i, ev in enumerate(eigenvalues):
    marker = "✓" if ev > 1 else "✗"
    print(f"  PC{i+1}: {ev:.3f} {marker}")


📊 INTERPRETATION:
Kaiser criterion suggests: 2 components (eigenvalue > 1)

Eigenvalues:
  PC1: 10.521 ✓
  PC2: 1.260 ✓
  PC3: 0.267 ✗
  PC4: 0.109 ✗
  PC5: 0.073 ✗
  PC6: 0.052 ✗
  PC7: 0.031 ✗
  PC8: 0.018 ✗
  PC9: 0.016 ✗
  PC10: 0.014 ✗
  PC11: 0.009 ✗
  PC12: 0.005 ✗


### 3.2 Explained Variance Plot

**Purpose:** Show individual and cumulative variance explained.

**Common thresholds:**
- 80% - adequate for exploration
- 90% - good for most purposes
- 95% - rigorous applications

In [45]:
components = np.arange(1, len(explained_var_ratio) + 1)
cumulative = np.cumsum(explained_var_ratio) * 100

fig = make_subplots(specs=[[{"secondary_y": True}]])

# Bar chart for individual variance
fig.add_trace(
    go.Bar(
        x=components,
        y=explained_var_ratio * 100,
        name='Individual',
        marker=dict(color=COLORS['score'], line=dict(color='black', width=1)),
        opacity=0.7,
        hovertemplate='PC%{x}<br>Individual: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=False
)

# Line for cumulative variance
fig.add_trace(
    go.Scatter(
        x=components,
        y=cumulative,
        mode='lines+markers',
        name='Cumulative',
        line=dict(color=COLORS['loading'], width=2),
        marker=dict(size=8),
        hovertemplate='PC%{x}<br>Cumulative: %{y:.1f}%<extra></extra>'
    ),
    secondary_y=True
)

# Threshold lines
fig.add_hline(y=80, line_dash='dash', line_color=COLORS['normal'], line_width=1.5, 
              opacity=0.7, secondary_y=True)
fig.add_hline(y=90, line_dash='dash', line_color=COLORS['warning'], line_width=1.5, 
              opacity=0.7, secondary_y=True)

# Add threshold annotations
fig.add_annotation(x=11.5, y=81, text='80%', showarrow=False, 
                   font=dict(size=10, color=COLORS['normal']), yref='y2')
fig.add_annotation(x=11.5, y=91, text='90%', showarrow=False, 
                   font=dict(size=10, color=COLORS['warning']), yref='y2')

# Set axis titles
fig.update_xaxes(title_text='Principal Component', tickmode='linear', tick0=1, dtick=1)
fig.update_yaxes(title_text='Variance Explained (%)', title_font=dict(color=COLORS['score']), 
                 tickfont=dict(color=COLORS['score']), secondary_y=False)
fig.update_yaxes(title_text='Cumulative Variance (%)', title_font=dict(color=COLORS['loading']), 
                 tickfont=dict(color=COLORS['loading']), range=[0, 105], secondary_y=True)

fig.update_layout(
    title='Explained Variance Plot',
    width=900,
    height=500,
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

# Find number of components for thresholds
n_80 = np.argmax(cumulative >= 80) + 1
n_90 = np.argmax(cumulative >= 90) + 1

print("\n📊 INTERPRETATION:")
print(f"Components needed for 80% variance: {n_80}")
print(f"Components needed for 90% variance: {n_90}")
print(f"\nRecommendation: Use {max(n_80, np.sum(eigenvalues > 1))} components")


📊 INTERPRETATION:
Components needed for 80% variance: 1
Components needed for 90% variance: 2

Recommendation: Use 2 components


### 3.3 Cross-Validation Plot

**Purpose:** Use cross-validation to find optimal number of components, avoiding overfitting.

In [46]:
from sklearn.model_selection import KFold

def compute_press(X, n_components, n_splits=5):
    """Compute PRESS using k-fold cross-validation."""
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    press = 0
    
    for train_idx, test_idx in kf.split(X):
        X_train = X[train_idx]
        X_test = X[test_idx]
        
        # Fit PCA on training data
        pca_cv = PCA(n_components=n_components)
        pca_cv.fit(X_train)
        
        # Reconstruct test data
        T_test = pca_cv.transform(X_test)
        X_reconstructed = pca_cv.inverse_transform(T_test)
        
        # Sum squared errors
        press += np.sum((X_test - X_reconstructed) ** 2)
    
    return press

# Compute PRESS for different numbers of components
max_components = min(X.shape[0] - 2, X.shape[1])  # Leave room for CV
press_values = []
for n in range(1, max_components + 1):
    press = compute_press(X, n)
    press_values.append(press)

# Convert to RMSECV
rmsecv = np.sqrt(np.array(press_values) / (X.shape[0] * X.shape[1]))

fig = go.Figure()

components = np.arange(1, max_components + 1)
min_idx = np.argmin(rmsecv)

# RMSECV line
fig.add_trace(go.Scatter(
    x=components,
    y=rmsecv,
    mode='lines+markers',
    line=dict(color=COLORS['score'], width=2),
    marker=dict(size=10),
    name='RMSECV',
    hovertemplate='PC%{x}<br>RMSECV: %{y:.4f}<extra></extra>'
))

# Mark minimum
fig.add_trace(go.Scatter(
    x=[components[min_idx]],
    y=[rmsecv[min_idx]],
    mode='markers',
    marker=dict(
        size=18,
        color=COLORS['loading'],
        line=dict(color='black', width=2)
    ),
    name=f'Minimum at PC{min_idx+1}',
    hovertemplate='Minimum<br>PC%{x}<br>RMSECV: %{y:.4f}<extra></extra>'
))

# Shade overfitting region
fig.add_vrect(
    x0=min_idx + 1.5,
    x1=max_components + 0.5,
    fillcolor=COLORS['loading'],
    opacity=0.2,
    line_width=0,
    annotation_text='Overfitting risk',
    annotation_position='top left'
)

fig.update_layout(
    title='Cross-Validation Plot (PRESS/RMSECV)',
    xaxis_title='Number of Components',
    yaxis_title='RMSECV',
    width=900,
    height=500,
    font=dict(size=12),
    xaxis=dict(tickmode='linear', tick0=1, dtick=1),
    hovermode='x unified'
)

fig.show()

print(f"\n📊 INTERPRETATION:")
print(f"Cross-validation suggests: {min_idx + 1} components (minimum RMSECV)")
print(f"Adding more components may lead to overfitting.")


📊 INTERPRETATION:
Cross-validation suggests: 12 components (minimum RMSECV)
Adding more components may lead to overfitting.


---
## 4. Outlier Detection & Diagnostics

Now let's choose our model (2 components based on the analysis above) and look for outliers.

In [43]:
# Set number of components for the model
r = 2  # Using 2 components

print(f"Model: {r} principal components")
print(f"Variance explained: {np.sum(explained_var_ratio[:r])*100:.1f}%")

Model: 2 principal components
Variance explained: 95.2%


### 4.1 Hotelling's T² Statistic

**Purpose:** Detect samples that are extreme *within* the model (unusual but following the pattern).

**Interpretation:**
- High T²: Sample has extreme scores (far from center in score space)
- But still follows the correlation structure captured by the model

In [52]:
# Compute T² and limit
T2 = compute_hotelling_t2(T, eigenvalues, r)
T2_limit = compute_t2_limit(X.shape[0], r, alpha=0.05)

# Sort by latitude for meaningful ordering
sort_idx = np.argsort(latitude)[::-1]
x_pos = np.arange(len(city_names))

# Separate outliers and normal samples
colors = [COLORS['outlier'] if t2 > T2_limit else COLORS['score'] for t2 in T2[sort_idx]]

fig = go.Figure()

# Bar chart
fig.add_trace(go.Bar(
    x=x_pos,
    y=T2[sort_idx],
    marker=dict(color=colors, line=dict(color='black', width=1)),
    opacity=0.8,
    text=city_names[sort_idx],
    hovertemplate='<b>%{text}</b><br>T²: %{y:.2f}<extra></extra>',
    showlegend=False
))

# Control limit
fig.add_hline(y=T2_limit, line_dash='dash', line_color=COLORS['loading'], line_width=2,
              annotation_text=f'95% limit ({T2_limit:.2f})', annotation_position='right')

fig.update_layout(
    title="Hotelling's T² Chart",
    xaxis_title='Cities (North → South)',
    yaxis_title="Hotelling's T²",
    width=1000,
    height=500,
    font=dict(size=12),
    xaxis=dict(
        tickmode='array',
        tickvals=x_pos,
        ticktext=city_names[sort_idx],
        tickangle=45
    ),
    hovermode='closest'
)

fig.show()

# List outliers
outliers_t2 = city_names[T2 > T2_limit]

print(f"\n📊 INTERPRETATION:")
print(f"T² control limit (95%): {T2_limit:.2f}")
if len(outliers_t2) > 0:
    print(f"Cities exceeding limit: {', '.join(outliers_t2)}")
    print("These cities have extreme but model-consistent temperature patterns.")
else:

    print("No cities exceed the T² limit - all fit well within the model.")


📊 INTERPRETATION:
T² control limit (95%): 6.82
Cities exceeding limit: Marbella, Moscow
These cities have extreme but model-consistent temperature patterns.


### 4.2 Q Statistic (SPE / DModX)

**Purpose:** Detect samples that don't fit the model structure.

**Interpretation:**
- High Q: Sample doesn't follow the correlation structure
- Could indicate measurement error, new behavior, or need for more components

In [53]:
# Compute Q and limit
Q, E = compute_q_statistic(X, T, P, r)
Q_limit = compute_q_limit(eigenvalues, r, alpha=0.05)

fig = go.Figure()

# Separate outliers and normal samples
colors = [COLORS['outlier'] if q > Q_limit else COLORS['normal'] for q in Q[sort_idx]]

# Bar chart
fig.add_trace(go.Bar(
    x=x_pos,
    y=Q[sort_idx],
    marker=dict(color=colors, line=dict(color='black', width=1)),
    opacity=0.8,
    text=city_names[sort_idx],
    hovertemplate='<b>%{text}</b><br>Q: %{y:.2f}<extra></extra>',
    showlegend=False
))

# Control limit
fig.add_hline(y=Q_limit, line_dash='dash', line_color=COLORS['loading'], line_width=2,
              annotation_text=f'95% limit ({Q_limit:.2f})', annotation_position='right')

fig.update_layout(
    title='Q Statistic (Squared Prediction Error) Chart',
    xaxis_title='Cities (North → South)',
    yaxis_title='Q (SPE)',
    width=1000,
    height=500,
    font=dict(size=12),
    xaxis=dict(
        tickmode='array',
        tickvals=x_pos,
        ticktext=city_names[sort_idx],
        tickangle=45
    ),
    hovermode='closest'
)

fig.show()

# List outliers
outliers_q = city_names[Q > Q_limit]

print(f"\n📊 INTERPRETATION:")
print(f"Q control limit (95%): {Q_limit:.2f}")
if len(outliers_q) > 0:
    print(f"Cities exceeding limit: {', '.join(outliers_q)}")
    print("These cities have temperature patterns not well captured by the 2-PC model.")
else:

    print("No cities exceed the Q limit - all patterns are well captured.")


📊 INTERPRETATION:
Q control limit (95%): 1.50
Cities exceeding limit: Vienna
These cities have temperature patterns not well captured by the 2-PC model.


### 4.3 T² vs Q Diagnostic Plot

**Purpose:** Combined view for outlier classification.

**Four quadrants:**
- **Normal (lower-left):** Typical sample, fits model well
- **High leverage (lower-right):** Extreme within model, influential
- **New type (upper-left):** Doesn't fit model but not extreme
- **Outlier (upper-right):** Extreme AND doesn't fit - investigate!

In [54]:
fig = go.Figure()

# Color code by quadrant and create separate traces for each quadrant
normal_x, normal_y, normal_text = [], [], []
high_lev_x, high_lev_y, high_lev_text = [], [], []
new_type_x, new_type_y, new_type_text = [], [], []
outlier_x, outlier_y, outlier_text = [], [], []

for i, (t2, q, city) in enumerate(zip(T2, Q, city_names)):
    if t2 <= T2_limit and q <= Q_limit:
        normal_x.append(t2)
        normal_y.append(q)
        normal_text.append(city)
    elif t2 > T2_limit and q <= Q_limit:
        high_lev_x.append(t2)
        high_lev_y.append(q)
        high_lev_text.append(city)
    elif t2 <= T2_limit and q > Q_limit:
        new_type_x.append(t2)
        new_type_y.append(q)
        new_type_text.append(city)
    else:
        outlier_x.append(t2)
        outlier_y.append(q)
        outlier_text.append(city)

# Add scatter traces for each quadrant
if normal_x:
    fig.add_trace(go.Scatter(
        x=normal_x, y=normal_y, text=normal_text,
        mode='markers+text', textposition='top right', textfont=dict(size=9),
        marker=dict(size=12, color=COLORS['normal'], line=dict(color='black', width=0.5)),
        name='Normal', hovertemplate='<b>%{text}</b><br>T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'
    ))

if high_lev_x:
    fig.add_trace(go.Scatter(
        x=high_lev_x, y=high_lev_y, text=high_lev_text,
        mode='markers+text', textposition='top right', textfont=dict(size=9),
        marker=dict(size=12, color=COLORS['warning'], line=dict(color='black', width=0.5)),
        name='High leverage', hovertemplate='<b>%{text}</b><br>T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'
    ))

if new_type_x:
    fig.add_trace(go.Scatter(
        x=new_type_x, y=new_type_y, text=new_type_text,
        mode='markers+text', textposition='top right', textfont=dict(size=9),
        marker=dict(size=12, color=COLORS['score'], line=dict(color='black', width=0.5)),
        name='New type', hovertemplate='<b>%{text}</b><br>T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'
    ))

if outlier_x:
    fig.add_trace(go.Scatter(
        x=outlier_x, y=outlier_y, text=outlier_text,
        mode='markers+text', textposition='top right', textfont=dict(size=9),
        marker=dict(size=12, color=COLORS['outlier'], line=dict(color='black', width=0.5)),
        name='Outlier', hovertemplate='<b>%{text}</b><br>T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'
    ))

# Draw quadrant lines
fig.add_hline(y=Q_limit, line_dash='dash', line_color=COLORS['loading'], line_width=2)
fig.add_vline(x=T2_limit, line_dash='dash', line_color=COLORS['loading'], line_width=2)

# Add quadrant labels
fig.add_annotation(x=T2_limit/2, y=Q_limit/2, text='NORMAL', showarrow=False,
                   font=dict(size=14, color=COLORS['normal']), opacity=0.5)
fig.add_annotation(x=T2_limit*1.5, y=Q_limit/2, text='HIGH<br>LEVERAGE', showarrow=False,
                   font=dict(size=12, color=COLORS['warning']), opacity=0.5)
fig.add_annotation(x=T2_limit/2, y=Q_limit*1.5, text='NEW<br>TYPE', showarrow=False,
                   font=dict(size=12, color=COLORS['score']), opacity=0.5)
fig.add_annotation(x=T2_limit*1.5, y=Q_limit*1.5, text='OUTLIER', showarrow=False,
                   font=dict(size=14, color=COLORS['outlier']), opacity=0.5)

fig.update_layout(
    title='T² vs Q Diagnostic Plot',
    xaxis_title="Hotelling's T²",
    yaxis_title='Q (SPE)',
    width=1000,
    height=800,
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

# Summarize findings
print("\n📊 SAMPLE CLASSIFICATION:")
for i, city in enumerate(city_names):
    if T2[i] <= T2_limit and Q[i] <= Q_limit:
        status = "Normal"
    elif T2[i] > T2_limit and Q[i] <= Q_limit:
        status = "⚠️ High leverage"
    elif T2[i] <= T2_limit and Q[i] > Q_limit:
        status = "⚠️ New type"
    else:
        status = "🚨 OUTLIER"
    
    if status != "Normal":
        print(f"  {city}: {status} (T²={T2[i]:.2f}, Q={Q[i]:.2f})")


📊 SAMPLE CLASSIFICATION:
  Marbella: ⚠️ High leverage (T²=8.48, Q=0.30)
  Moscow: ⚠️ High leverage (T²=7.73, Q=0.83)
  Vienna: ⚠️ New type (T²=1.15, Q=5.60)


### 4.4 Influence Plot (Leverage vs Residual)

**Purpose:** Identify samples that have high influence on the model.

In [55]:
# Compute leverage (hat values)
T_r = T[:, :r]
leverage = np.diag(T_r @ np.linalg.inv(T_r.T @ T_r) @ T_r.T)

# Residual magnitude
residual_norm = np.sqrt(np.sum(E ** 2, axis=1))

# Threshold for leverage (2p/n)
leverage_limit = 2 * r / X.shape[0]

fig = go.Figure()

# Scatter plot
fig.add_trace(go.Scatter(
    x=leverage,
    y=residual_norm,
    mode='markers+text',
    text=city_names,
    textposition='top right',
    textfont=dict(size=9),
    marker=dict(
        size=10,
        color=COLORS['score'],
        line=dict(color='black', width=1)
    ),
    hovertemplate='<b>%{text}</b><br>Leverage: %{x:.3f}<br>Residual: %{y:.2f}<extra></extra>'
))

# Threshold line
fig.add_vline(x=leverage_limit, line_dash='dash', line_color=COLORS['loading'], line_width=2,
              annotation_text=f'Leverage limit (2p/n = {leverage_limit:.3f})', 
              annotation_position='top right')

fig.update_layout(
    title='Influence Plot',
    xaxis_title='Leverage (hat value)',
    yaxis_title='Residual Magnitude',
    width=900,
    height=700,
    font=dict(size=12),
    hovermode='closest'
)

fig.show()

# List influential points
influential = city_names[leverage > leverage_limit]
print(f"\n📊 INTERPRETATION:")
print(f"Leverage limit: {leverage_limit:.3f}")
if len(influential) > 0:
    print(f"Influential cities: {', '.join(influential)}")
else:
    print("No highly influential cities identified.")


📊 INTERPRETATION:
Leverage limit: 0.121
Influential cities: Athens, Dublin, Marbella, Moscow, St. Petersberg


---
## 5. Fault Diagnosis / Root Cause

When we identify outliers, contribution plots help us understand *which variables* are responsible.

### 5.1 T² Contribution Plot

**Purpose:** Identify which variables contribute most to a sample's high T².

In [56]:
# Select a city to analyze (choose one with interesting T² or highest T²)
sample_idx = np.argmax(T2)  # Highest T² sample
sample_name = city_names[sample_idx]

# Compute T² contributions
T2_contrib = compute_t2_contributions(X, T, P, eigenvalues, r)
sample_contrib = T2_contrib[sample_idx]

fig = go.Figure()

# Separate positive and negative contributions
colors = [COLORS['loading'] if c > 0 else COLORS['score'] for c in sample_contrib]

# Bar chart
fig.add_trace(go.Bar(
    x=month_cols,
    y=sample_contrib,
    marker=dict(color=colors, line=dict(color='black', width=1)),
    opacity=0.8,
    hovertemplate='<b>%{x}</b><br>Contribution: %{y:.3f}<extra></extra>',
    showlegend=False
))

# Reference line at 0
fig.add_hline(y=0, line_color='black', line_width=0.8)

# Highlight main contributor
main_contrib_idx = np.argmax(np.abs(sample_contrib))
fig.add_annotation(
    x=month_cols[main_contrib_idx],
    y=sample_contrib[main_contrib_idx],
    text='Main contributor',
    showarrow=True,
    arrowhead=2,
    arrowcolor=COLORS['outlier'],
    ax=40,
    ay=-30 if sample_contrib[main_contrib_idx] > 0 else 30,
    font=dict(size=10, color=COLORS['outlier'])
)

fig.update_layout(
    title=f'T² Contribution Plot for {sample_name} (T²={T2[sample_idx]:.2f})',
    xaxis_title='Variable (Month)',
    yaxis_title='T² Contribution',
    width=900,
    height=500,
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

sorted_idx = np.argsort(np.abs(sample_contrib))[::-1]
print(f"\n📊 INTERPRETATION for {sample_name}:")
print("Top 3 contributing months:")
for i in range(3):
    idx = sorted_idx[i]
    print(f"  {month_cols[idx]}: {sample_contrib[idx]:.3f}")


📊 INTERPRETATION for Marbella:
Top 3 contributing months:
  Jan: 1.025
  Dec: 1.010
  Nov: 0.998


### 5.2 Q Contribution Plot

**Purpose:** Identify which variables don't fit the model for a specific sample.

In [62]:
# Select sample with highest Q
sample_idx = np.argmax(Q)
sample_name = city_names[sample_idx]

# Q contributions are squared residuals
Q_contrib = compute_q_contributions(E)
sample_Q_contrib = Q_contrib[sample_idx]

fig = go.Figure()

# Color by relative magnitude
max_contrib = np.max(sample_Q_contrib)
colors = [COLORS['outlier'] if c > max_contrib * 0.5 else COLORS['score'] for c in sample_Q_contrib]

# Bar chart
fig.add_trace(go.Bar(
    x=month_cols,
    y=sample_Q_contrib,
    marker=dict(color=colors, line=dict(color='black', width=1)),
    opacity=0.8,
    hovertemplate='<b>%{x}</b><br>Q Contribution: %{y:.3f}<extra></extra>',
    showlegend=False
))

fig.update_layout(
    title=f'Q Contribution Plot for {sample_name} (Q={Q[sample_idx]:.2f})',
    xaxis_title='Variable (Month)',
    yaxis_title='Q Contribution (e²)',
    width=900,
    height=500,
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

sorted_idx = np.argsort(sample_Q_contrib)[::-1]
print(f"\n📊 INTERPRETATION for {sample_name}:")
print("Variables with highest residuals (poorest fit):")
for i in range(3):
    idx = sorted_idx[i]
    print(f"  {month_cols[idx]}: e² = {sample_Q_contrib[idx]:.3f}")


📊 INTERPRETATION for Vienna:
Variables with highest residuals (poorest fit):
  Jun: e² = 2.309
  Jul: e² = 0.912
  Aug: e² = 0.794


### 5.3 Score Contribution Plot

**Purpose:** Understand why a sample has a specific score value.

In [63]:
# Compare two contrasting cities
city1 = 'Helsinki'  # Cold northern city
city2 = 'Athens'    # Warm southern city

idx1 = np.where(city_names == city1)[0][0]
idx2 = np.where(city_names == city2)[0][0]

# Score contributions: X_scaled * loading
score_contrib_1 = X[idx1] * P[:, 0]  # Contribution to PC1
score_contrib_2 = X[idx2] * P[:, 0]

# Create subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=(f'{city1} (PC1 = {T[idx1, 0]:.2f})', 
                                                     f'{city2} (PC1 = {T[idx2, 0]:.2f})'))

# City 1
colors1 = [COLORS['score'] if c > 0 else COLORS['loading'] for c in score_contrib_1]
fig.add_trace(
    go.Bar(
        x=month_cols,
        y=score_contrib_1,
        marker=dict(color=colors1, line=dict(color='black', width=1)),
        opacity=0.8,
        name=city1,
        hovertemplate='<b>%{x}</b><br>Contribution: %{y:.3f}<extra></extra>'
    ),
    row=1, col=1
)

# City 2
colors2 = [COLORS['score'] if c > 0 else COLORS['loading'] for c in score_contrib_2]
fig.add_trace(
    go.Bar(
        x=month_cols,
        y=score_contrib_2,
        marker=dict(color=colors2, line=dict(color='black', width=1)),
        opacity=0.8,
        name=city2,
        hovertemplate='<b>%{x}</b><br>Contribution: %{y:.3f}<extra></extra>'
    ),
    row=1, col=2
)

# Add reference lines
fig.add_hline(y=0, line_color='black', line_width=0.8, row=1, col=1)
fig.add_hline(y=0, line_color='black', line_width=0.8, row=1, col=2)

# Update axes
fig.update_xaxes(title_text='Month', tickangle=45, row=1, col=1)
fig.update_xaxes(title_text='Month', tickangle=45, row=1, col=2)
fig.update_yaxes(title_text='Contribution to PC1', row=1, col=1)
fig.update_yaxes(title_text='Contribution to PC1', row=1, col=2)

fig.update_layout(
    title_text='Score Contribution Comparison',
    width=1100,
    height=450,
    showlegend=False,
    font=dict(size=11),
    hovermode='x unified'
)

fig.show()

print("\n📊 INTERPRETATION:")
print(f"{city1} has negative PC1 score because all months contribute negatively")
print(f"  (below-average temperatures throughout the year)")
print(f"{city2} has positive PC1 score because all months contribute positively")
print(f"  (above-average temperatures throughout the year)")


📊 INTERPRETATION:
Helsinki has negative PC1 score because all months contribute negatively
  (below-average temperatures throughout the year)
Athens has positive PC1 score because all months contribute positively
  (above-average temperatures throughout the year)


---
## 6. Variable Analysis

These plots help us understand variable importance and relationships.

### 6.1 Loading Bar Chart

**Purpose:** Interpret individual principal components by examining their loadings.

In [64]:
# Create subplots for PC1 and PC2
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'PC1 Loadings ({explained_var_ratio[0]*100:.1f}% variance)', 
    f'PC2 Loadings ({explained_var_ratio[1]*100:.1f}% variance)'))

for pc_idx in range(2):
    loadings = P[:, pc_idx]
    colors = [COLORS['score'] if l > 0 else COLORS['loading'] for l in loadings]
    
    fig.add_trace(
        go.Bar(
            x=month_cols,
            y=loadings,
            marker=dict(color=colors, line=dict(color='black', width=1)),
            opacity=0.8,
            name=f'PC{pc_idx+1}',
            hovertemplate='<b>%{x}</b><br>Loading: %{y:.3f}<extra></extra>'
        ),
        row=1, col=pc_idx+1
    )
    
    # Add reference line at 0
    fig.add_hline(y=0, line_color='black', line_width=0.8, row=1, col=pc_idx+1)

# Update axes
fig.update_xaxes(title_text='Month', tickangle=45, row=1, col=1)
fig.update_xaxes(title_text='Month', tickangle=45, row=1, col=2)
fig.update_yaxes(title_text='Loading', row=1, col=1)
fig.update_yaxes(title_text='Loading', row=1, col=2)

fig.update_layout(
    width=1100,
    height=450,
    showlegend=False,
    font=dict(size=11),
    hovermode='x unified'
)

fig.show()

print("\n📊 INTERPRETATION:")
print("PC1: All months load positively → represents 'overall temperature level'")
print("PC2: Winter months (Dec-Feb) load positively, Summer (Jun-Aug) negatively")
print("  → represents 'seasonality' or 'continentality' of climate")


📊 INTERPRETATION:
PC1: All months load positively → represents 'overall temperature level'
PC2: Winter months (Dec-Feb) load positively, Summer (Jun-Aug) negatively
  → represents 'seasonality' or 'continentality' of climate


### 6.2 Loading Heatmap

**Purpose:** Visualize all loadings across multiple components simultaneously.

In [60]:
# Create loading matrix for heatmap
loading_matrix = P[:, :5]  # First 5 PCs
pc_labels = [f'PC{i+1}<br>({explained_var_ratio[i]*100:.1f}%)' for i in range(5)]

fig = go.Figure()

# Create heatmap
fig.add_trace(go.Heatmap(
    z=loading_matrix,
    x=pc_labels,
    y=month_cols,
    colorscale='RdBu_r',
    zmid=0,
    text=np.round(loading_matrix, 2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title='Loading value'),
    hovertemplate='<b>%{y}</b><br>%{x}<br>Loading: %{z:.3f}<extra></extra>'
))

fig.update_layout(
    title='Loading Heatmap',
    xaxis_title='Principal Component',
    yaxis_title='Variable (Month)',
    width=900,
    height=700,
    font=dict(size=12)
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- Red: positive loading (variable increases with PC)")
print("- Blue: negative loading (variable decreases with PC)")
print("- PC1: uniform positive loadings (overall temperature)")
print("- PC2: seasonal contrast (winter vs summer)")


📊 INTERPRETATION:
- Red: positive loading (variable increases with PC)
- Blue: negative loading (variable decreases with PC)
- PC1: uniform positive loadings (overall temperature)
- PC2: seasonal contrast (winter vs summer)


### 6.3 Variable Importance Plot

**Purpose:** Rank variables by their overall contribution to the model.

In [61]:
# Compute variable importance (weighted by explained variance)
importance = np.sum(P[:, :r]**2 * eigenvalues[:r], axis=1)
importance_normalized = importance / np.sum(importance) * 100

# Sort by importance
sort_idx = np.argsort(importance_normalized)[::-1]

fig = go.Figure()

# Horizontal bar chart
fig.add_trace(go.Bar(
    x=importance_normalized[sort_idx],
    y=np.array(month_cols)[sort_idx],
    orientation='h',
    marker=dict(color=COLORS['score'], line=dict(color='black', width=1)),
    opacity=0.8,
    text=[f'{val:.1f}%' for val in importance_normalized[sort_idx]],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Importance: %{x:.1f}%<extra></extra>'
))

fig.update_layout(
    title='Variable Importance in PCA Model',
    xaxis_title='Importance Score (%)',
    yaxis_title='Variable (Month)',
    width=900,
    height=500,
    font=dict(size=12),
    showlegend=False
)

fig.show()

print("\n📊 INTERPRETATION:")
print("Variables are roughly equally important (expected for temperature data).")
print("Slight variations show seasonal weighting in the model.")


📊 INTERPRETATION:
Variables are roughly equally important (expected for temperature data).
Slight variations show seasonal weighting in the model.


### 6.4 Communalities Plot

**Purpose:** Show how much variance of each variable is explained by the model.

In [71]:
# Communalities = sum of squared loadings
communalities = np.sum(P[:, :r]**2, axis=1)

fig = go.Figure()

# Separate well and poorly explained variables
colors = [COLORS['normal'] if c > 0.5 else COLORS['warning'] for c in communalities]

# Bar chart
fig.add_trace(go.Bar(
    x=month_cols,
    y=communalities,
    marker=dict(color=colors, line=dict(color='black', width=1)),
    opacity=0.8,
    text=[f'{val:.2f}' for val in communalities],
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Communality: %{y:.3f}<extra></extra>',
    showlegend=False
))

# 50% threshold
fig.add_hline(y=0.5, line_dash='dash', line_color=COLORS['loading'], line_width=2,
              annotation_text='50% threshold', annotation_position='right')

fig.update_layout(
    title=f'Communalities (r={r} components)',
    xaxis_title='Variable (Month)',
    yaxis_title='Communality (Variance Explained)',
    width=900,
    height=500,
    font=dict(size=12),
    yaxis=dict(range=[0, 1.1])
)

fig.show()

low_communality = np.array(month_cols)[communalities < 0.5]
print("\n📊 INTERPRETATION:")
if len(low_communality) > 0:
    print(f"Variables with low communality (<50%): {', '.join(low_communality)}")
    print("These months may need additional PCs to be well represented.")
else:
    print(f"All variables have communality > 50% with {r} components.")


📊 INTERPRETATION:
Variables with low communality (<50%): Jan, Feb, Mar, Apr, May, Jun, Jul, Aug, Sep, Oct, Nov, Dec
These months may need additional PCs to be well represented.


---
## 7. Residual Analysis

These plots assess the quality of the model fit.

### 7.1 Residual Matrix Heatmap

**Purpose:** Visualize the pattern of residuals across all samples and variables.

In [72]:
# Sort cities by latitude for meaningful patterns
sort_idx = np.argsort(latitude)[::-1]
E_sorted = E[sort_idx]

fig = go.Figure()

# Create heatmap
fig.add_trace(go.Heatmap(
    z=E_sorted,
    x=month_cols,
    y=city_names[sort_idx],
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(title='Residual'),
    hovertemplate='<b>%{y}</b><br>%{x}<br>Residual: %{z:.3f}<extra></extra>'
))

fig.update_layout(
    title='Residual Matrix Heatmap',
    xaxis_title='Variable (Month)',
    yaxis_title='City (North → South)',
    width=1000,
    height=800,
    font=dict(size=12)
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- Random pattern: good model fit")
print("- Systematic rows/columns: poor fit for specific cities/months")
print("- Look for 'hot spots' indicating model deficiencies")


📊 INTERPRETATION:
- Random pattern: good model fit
- Systematic rows/columns: poor fit for specific cities/months
- Look for 'hot spots' indicating model deficiencies


### 7.2 Predicted vs Actual Plot

**Purpose:** Assess reconstruction quality for each variable.

In [73]:
# Reconstruct data
X_reconstructed = T[:, :r] @ P[:, :r].T

# Select a few months to show
months_to_plot = ['Jan', 'Apr', 'Jul', 'Oct']

# Create subplots
fig = make_subplots(rows=2, cols=2, subplot_titles=months_to_plot)

for i, month in enumerate(months_to_plot):
    row = i // 2 + 1
    col = i % 2 + 1
    
    idx = month_cols.index(month)
    actual = X[:, idx]
    predicted = X_reconstructed[:, idx]
    
    # Calculate R²
    ss_res = np.sum((actual - predicted) ** 2)
    ss_tot = np.sum((actual - np.mean(actual)) ** 2)
    r2 = 1 - ss_res / ss_tot
    
    # Scatter plot
    fig.add_trace(
        go.Scatter(
            x=actual,
            y=predicted,
            mode='markers',
            marker=dict(size=8, color=COLORS['score'], line=dict(color='black', width=0.5)),
            opacity=0.7,
            name=f'{month} (R²={r2:.3f})',
            hovertemplate='Actual: %{x:.2f}<br>Predicted: %{y:.2f}<extra></extra>'
        ),
        row=row, col=col
    )
    
    # Perfect fit line
    lims = [min(actual.min(), predicted.min()) - 0.5, max(actual.max(), predicted.max()) + 0.5]
    fig.add_trace(
        go.Scatter(
            x=lims,
            y=lims,
            mode='lines',
            line=dict(color=COLORS['loading'], width=2, dash='dash'),
            showlegend=False,
            hoverinfo='skip'
        ),
        row=row, col=col
    )
    
    # Update axes
    fig.update_xaxes(title_text='Actual (standardized)', range=lims, scaleanchor=f'y{i+1}', scaleratio=1, row=row, col=col)
    fig.update_yaxes(title_text='Predicted (standardized)', range=lims, row=row, col=col)

fig.update_layout(
    title_text='Predicted vs Actual Plots',
    width=1000,
    height=800,
    font=dict(size=10),
    showlegend=False
)

fig.show()

print("\n📊 INTERPRETATION:")
print("Points on the diagonal line indicate perfect predictions.")
print("Scatter around the line shows prediction error.")
print(f"R² close to 1 means good reconstruction with {r} PCs.")


📊 INTERPRETATION:
Points on the diagonal line indicate perfect predictions.
Scatter around the line shows prediction error.
R² close to 1 means good reconstruction with 2 PCs.


### 7.3 Residual Histogram

**Purpose:** Check if residuals are normally distributed (as expected for a good model).

In [74]:
# Flatten residuals
residuals_flat = E.flatten()

# Overlay normal distribution
mu, std = np.mean(residuals_flat), np.std(residuals_flat)
x = np.linspace(residuals_flat.min(), residuals_flat.max(), 100)
normal_pdf = stats.norm.pdf(x, mu, std)

fig = go.Figure()

# Histogram
fig.add_trace(go.Histogram(
    x=residuals_flat,
    nbinsx=30,
    histnorm='probability density',
    marker=dict(color=COLORS['score'], line=dict(color='black', width=1)),
    opacity=0.7,
    name='Residuals',
    hovertemplate='Value: %{x:.2f}<br>Density: %{y:.4f}<extra></extra>'
))

# Overlay normal distribution
fig.add_trace(go.Scatter(
    x=x,
    y=normal_pdf,
    mode='lines',
    line=dict(color=COLORS['loading'], width=3),
    name=f'Normal (μ={mu:.2f}, σ={std:.2f})',
    hovertemplate='Value: %{x:.2f}<br>Density: %{y:.4f}<extra></extra>'
))

# Reference line at 0
fig.add_vline(x=0, line_dash='dash', line_color='black', line_width=1.5)

fig.update_layout(
    title='Residual Distribution',
    xaxis_title='Residual Value',
    yaxis_title='Density',
    width=900,
    height=500,
    font=dict(size=12),
    hovermode='x unified'
)

fig.show()

# Normality test
statistic, p_value = stats.shapiro(residuals_flat[:50])  # Shapiro-Wilk test (sample)
print(f"\n📊 INTERPRETATION:")
print(f"Shapiro-Wilk test: statistic={statistic:.4f}, p-value={p_value:.4f}")
if p_value > 0.05:
    print("Residuals appear normally distributed (good model fit).")
else:
    print("Residuals may deviate from normality (investigate further).")



📊 INTERPRETATION:
Shapiro-Wilk test: statistic=0.9700, p-value=0.2315
Residuals appear normally distributed (good model fit).


### 7.4 Q-Q Plot

**Purpose:** More sensitive test for normality of residuals.

In [ ]:
# Calculate Q-Q plot data
(osm, osr), (slope, intercept, r_corr) = stats.probplot(residuals_flat, dist="norm")

fig = go.Figure()

# Theoretical line
fig.add_trace(go.Scatter(
    x=osm,
    y=slope * osm + intercept,
    mode='lines',
    line=dict(color=COLORS['loading'], width=2),
    name='Theoretical line',
    hoverinfo='skip'
))

# Sample quantiles
fig.add_trace(go.Scatter(
    x=osm,
    y=osr,
    mode='markers',
    marker=dict(size=8, color=COLORS['score']),
    name='Sample quantiles',
    hovertemplate='Theoretical: %{x:.2f}<br>Sample: %{y:.2f}<extra></extra>'
))

fig.update_layout(
    title='Q-Q Plot of Residuals',
    xaxis_title='Theoretical Quantiles',
    yaxis_title='Sample Quantiles',
    width=700,
    height=700,
    font=dict(size=12)
)

fig.show()

print("\n📊 INTERPRETATION:")
print("- Points on the line: residuals are normally distributed")
print("- S-curve: heavy or light tails")
print("- Curved pattern: skewness")
print("- Points off the line at extremes: outliers")


📊 INTERPRETATION:
- Points on the line: residuals are normally distributed
- S-curve: heavy or light tails
- Curved pattern: skewness
- Points off the line at extremes: outliers


---
## 8. Summary Dashboard

A comprehensive view of key diagnostic plots in one figure.

In [ ]:
# Create 2x3 subplot dashboard
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Score Plot', 'Loading Plot', 'Scree Plot', 
                    'T² vs Q Plot', 'Explained Variance', 'Loading Heatmap'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}],
           [{'type': 'scatter'}, {'type': 'xy', 'secondary_y': True}, {'type': 'heatmap'}]]
)

# 1. Score plot (top-left)
fig.add_trace(
    go.Scatter(
        x=T[:, 0], y=T[:, 1],
        mode='markers',
        marker=dict(size=6, color=latitude, colorscale='RdBu_r', showscale=True,
                    colorbar=dict(x=0.31, y=0.85, len=0.3, title='Lat'),
                    line=dict(color='black', width=0.5)),
        showlegend=False,
        hovertemplate='PC1: %{x:.2f}<br>PC2: %{y:.2f}<extra></extra>'
    ),
    row=1, col=1
)
fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.5, row=1, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='gray', line_width=0.5, row=1, col=1)

# 2. Loading plot (top-center)
for i, month in enumerate(month_cols):
    fig.add_trace(go.Scatter(x=[0, P[i, 0]], y=[0, P[i, 1]], mode='lines',
                             line=dict(color=COLORS['loading'], width=1.5),
                             showlegend=False, hoverinfo='skip'), row=1, col=2)
    fig.add_annotation(x=P[i, 0], y=P[i, 1], text=month, showarrow=False,
                       font=dict(size=7), row=1, col=2, xref='x2', yref='y2')
fig.add_hline(y=0, line_color='gray', line_width=0.5, row=1, col=2)
fig.add_vline(x=0, line_color='gray', line_width=0.5, row=1, col=2)

# 3. Scree plot (top-right)
fig.add_trace(
    go.Scatter(x=list(range(1, len(eigenvalues)+1)), y=eigenvalues, mode='lines+markers',
               line=dict(color=COLORS['score']), marker=dict(size=6),
               showlegend=False, hovertemplate='PC%{x}<br>λ: %{y:.2f}<extra></extra>'),
    row=1, col=3
)
fig.add_hline(y=1, line_dash='dash', line_color=COLORS['loading'], line_width=1.5, row=1, col=3)

# 4. T² vs Q (bottom-left)
dashboard_colors = [COLORS['outlier'] if (t2 > T2_limit or q > Q_limit) else COLORS['normal'] 
                    for t2, q in zip(T2, Q)]
fig.add_trace(
    go.Scatter(x=T2, y=Q, mode='markers',
               marker=dict(size=6, color=dashboard_colors, line=dict(color='black', width=0.5)),
               showlegend=False, hovertemplate='T²: %{x:.2f}<br>Q: %{y:.2f}<extra></extra>'),
    row=2, col=1
)
fig.add_hline(y=Q_limit, line_dash='dash', line_color=COLORS['loading'], line_width=1.5, row=2, col=1)
fig.add_vline(x=T2_limit, line_dash='dash', line_color=COLORS['loading'], line_width=1.5, row=2, col=1)

# 5. Explained variance (bottom-center) - Bars on primary, line on secondary
fig.add_trace(
    go.Bar(x=list(range(1, len(explained_var_ratio)+1)), y=explained_var_ratio*100,
           marker=dict(color=COLORS['score'], line=dict(color='black', width=1)), opacity=0.7,
           showlegend=False, hovertemplate='PC%{x}<br>Var: %{y:.1f}%<extra></extra>'),
    row=2, col=2, secondary_y=False
)
fig.add_trace(
    go.Scatter(x=list(range(1, len(explained_var_ratio)+1)), y=np.cumsum(explained_var_ratio)*100,
               mode='lines+markers', line=dict(color=COLORS['loading']), marker=dict(size=4),
               showlegend=False, hovertemplate='PC%{x}<br>Cum: %{y:.1f}%<extra></extra>'),
    row=2, col=2, secondary_y=True
)
fig.add_hline(y=90, line_dash='dash', line_color=COLORS['warning'], line_width=1, 
              row=2, col=2, secondary_y=True)

# 6. Loading heatmap (bottom-right)
fig.add_trace(
    go.Heatmap(z=P[:, :4], x=[f'PC{i+1}' for i in range(4)], y=month_cols,
               colorscale='RdBu_r', zmid=0, showscale=False,
               text=np.round(P[:, :4], 2), texttemplate='%{text}', textfont=dict(size=7),
               hovertemplate='%{y}<br>%{x}<br>%{z:.3f}<extra></extra>'),
    row=2, col=3
)

# Update axes labels
fig.update_xaxes(title_text=f'PC1 ({explained_var_ratio[0]*100:.1f}%)', row=1, col=1)
fig.update_yaxes(title_text=f'PC2 ({explained_var_ratio[1]*100:.1f}%)', row=1, col=1)
fig.update_xaxes(title_text='PC1 Loading', range=[-0.5, 0.5], row=1, col=2)
fig.update_yaxes(title_text='PC2 Loading', range=[-0.8, 0.8], row=1, col=2)
fig.update_xaxes(title_text='Component', row=1, col=3)
fig.update_yaxes(title_text='Eigenvalue', row=1, col=3)
fig.update_xaxes(title_text="T²", row=2, col=1)
fig.update_yaxes(title_text='Q', row=2, col=1)
fig.update_xaxes(title_text='Component', row=2, col=2)
fig.update_yaxes(title_text='Var (%)', row=2, col=2, secondary_y=False)
fig.update_yaxes(title_text='Cum (%)', row=2, col=2, secondary_y=True)

fig.update_layout(
    title_text='PCA Diagnostic Dashboard: European City Temperatures',
    title_font_size=16,
    width=1400,
    height=900,
    font=dict(size=10),
    showlegend=False
)

fig.show()

print("\n" + "="*70)
print("SUMMARY OF KEY FINDINGS")
print("="*70)
print(f"\n1. MODEL: {n_components} principal components explain {np.sum(explained_var_ratio[:n_components])*100:.1f}% of variance")
print(f"\n2. PC1 ({explained_var_ratio[0]*100:.1f}%): Overall temperature level (latitude effect)")
print(f"   PC2 ({explained_var_ratio[1]*100:.1f}%): Seasonality (continental vs maritime climate)")
print(f"\n3. OUTLIERS: {np.sum(T2 > T2_limit)} cities exceed T² limit, {np.sum(Q > Q_limit)} exceed Q limit")
print(f"\n4. Geographic pattern: Northern cities (low PC1) vs Southern cities (high PC1)")


SUMMARY OF KEY FINDINGS

1. MODEL: 12 principal components explain 100.0% of variance

2. PC1 (85.0%): Overall temperature level (latitude effect)
   PC2 (10.2%): Seasonality (continental vs maritime climate)

3. OUTLIERS: 2 cities exceed T² limit, 1 exceed Q limit

4. All months are well-represented (communalities > 0.5)

5. Geographic pattern: Northern cities (low PC1) vs Southern cities (high PC1)


---
## Key Takeaways

### When to use each plot:

| Goal | Primary Plot | Supporting Plots |
|------|--------------|------------------|
| Explore sample relationships | Score plot | Biplot, 3D scores |
| Understand variables | Loading plot | Correlation loadings, heatmap |
| Select # components | Scree plot | Cross-validation, explained variance |
| Find outliers | T² vs Q plot | Control charts, influence plot |
| Diagnose faults | Contribution plots | Score contributions |
| Assess model quality | Residual histogram | Q-Q plot, predicted vs actual |

### Recommended workflow:

1. **Validate**: Use scree/CV plots to select number of components
2. **Interpret**: Score + Loading plots to understand structure  
3. **Diagnose**: T²/Q plots to find outliers
4. **Root cause**: Contribution plots to understand outliers
5. **Verify**: Residual plots to check model assumptions